<a href="https://colab.research.google.com/github/ClickDgo/Dashboard-Defunciones/blob/main/Proyecto_Maestr%C3%ADa_(Dise%C3%B1o_de_experimentos).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS DE DEFUNCIONES
# Análisis general y por causa
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTAR LIBRERÍAS
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files
from IPython.display import display, HTML
import io
import warnings

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. CARGAR ARCHIVO CSV
# ------------------------------------------------------------

print("Selecciona el archivo CSV...")

uploaded = files.upload()

# Obtener nombre del archivo
nombre_archivo = list(uploaded.keys())[0]

print(f"\nArchivo cargado: {nombre_archivo}")


# ------------------------------------------------------------
# 3. LEER CSV
# ------------------------------------------------------------

# Intentar diferentes codificaciones
codificaciones = ["utf-8", "latin1", "cp1252"]

datos = None

for enc in codificaciones:
    try:
        datos = pd.read_csv(
            io.BytesIO(uploaded[nombre_archivo]),
            encoding=enc
        )
        print(f"Codificación utilizada: {enc}")
        break
    except UnicodeDecodeError:
        continue

if datos is None:
    raise ValueError("No fue posible leer el archivo con las codificaciones disponibles.")


# ------------------------------------------------------------
# 4. LIMPIEZA BÁSICA DE NOMBRES Y TEXTO
# ------------------------------------------------------------

datos.columns = datos.columns.str.strip()

columnas_texto = datos.select_dtypes(include="object").columns

for col in columnas_texto:
    datos[col] = (
        datos[col]
        .astype(str)
        .str.strip()
    )

# Convertir algunos valores que pudieran haber sido interpretados
# como texto "nan"
datos = datos.replace(["nan", "NaN", "None", ""], np.nan)


# ------------------------------------------------------------
# 5. VERIFICACIÓN DE COLUMNAS
# ------------------------------------------------------------

columnas_necesarias = [
    "Causa",
    "Sexo",
    "AfroMexicanos",
    "CondicionIndigena",
    "Edad",
    "Escolaridad",
    "Ocupacion",
    "Ocupacion_Ampliado",
    "EstadoCivil"
]

faltantes = [
    col for col in columnas_necesarias
    if col not in datos.columns
]

if faltantes:
    raise ValueError(
        "Faltan las siguientes columnas en el archivo:\n"
        + "\n".join(f"- {x}" for x in faltantes)
    )


# ------------------------------------------------------------
# 6. INFORMACIÓN GENERAL DE LA BASE
# ------------------------------------------------------------

N_TOTAL = len(datos)

numero_causas = datos["Causa"].nunique()

causas = (
    datos["Causa"]
    .dropna()
    .value_counts()
    .index
    .tolist()
)


# ------------------------------------------------------------
# 7. FUNCIONES PARA TABLAS CATEGÓRICAS
# ------------------------------------------------------------

def tabla_categorica(df, columna):

    serie = df[columna].dropna()

    tabla = (
        serie
        .value_counts()
        .rename_axis("Categoría")
        .reset_index(name="Total")
    )

    tabla["Porcentaje"] = (
        tabla["Total"] / tabla["Total"].sum() * 100
    )

    tabla["Porcentaje"] = tabla["Porcentaje"].round(2)

    # Agregar total
    fila_total = pd.DataFrame({
        "Categoría": ["TOTAL"],
        "Total": [tabla["Total"].sum()],
        "Porcentaje": [100.00]
    })

    tabla = pd.concat(
        [tabla, fila_total],
        ignore_index=True
    )

    return tabla


# ------------------------------------------------------------
# 8. FUNCIÓN PARA ESTADÍSTICAS DE EDAD
# ------------------------------------------------------------

def estadisticas_edad(df):

    edad = pd.to_numeric(
        df["Edad"],
        errors="coerce"
    ).dropna()

    n = len(edad)

    if n == 0:
        return None

    q1 = edad.quantile(0.25)
    mediana = edad.quantile(0.50)
    q3 = edad.quantile(0.75)

    media = edad.mean()
    desviacion = edad.std()
    minimo = edad.min()
    maximo = edad.max()

    rango = maximo - minimo
    iqr = q3 - q1

    asimetria = stats.skew(edad, bias=False)
    curtosis = stats.kurtosis(
        edad,
        fisher=True,
        bias=False
    )

    # --------------------------------------------------------
    # Prueba de normalidad para muestras grandes
    # D'Agostino-Pearson
    # --------------------------------------------------------

    # La prueba requiere al menos 8 observaciones
    if n >= 8:

        statistic_normalidad, p_normalidad = (
            stats.normaltest(edad)
        )

        if p_normalidad < 0.05:
            conclusion_normalidad = (
                "Se rechaza la normalidad"
            )
        else:
            conclusion_normalidad = (
                "No se rechaza la normalidad"
            )

    else:

        statistic_normalidad = np.nan
        p_normalidad = np.nan

        conclusion_normalidad = (
            "No es posible aplicar la prueba"
        )

    resumen = pd.DataFrame({
        "Estadístico": [
            "N",
            "Media",
            "Desviación estándar",
            "Mínimo",
            "Q1",
            "Mediana",
            "Q3",
            "Máximo",
            "Rango",
            "IQR",
            "Asimetría",
            "Curtosis",
            "D'Agostino-Pearson",
            "p-valor normalidad"
        ],

        "Valor": [
            n,
            media,
            desviacion,
            minimo,
            q1,
            mediana,
            q3,
            maximo,
            rango,
            iqr,
            asimetria,
            curtosis,
            statistic_normalidad,
            p_normalidad
        ]
    })

    return resumen, conclusion_normalidad


# ------------------------------------------------------------
# 9. FUNCIÓN PARA TABLA DE DISTRIBUCIÓN DE EDADES
# ------------------------------------------------------------

def tabla_edades(df):

    edad = pd.to_numeric(
        df["Edad"],
        errors="coerce"
    ).dropna()

    tabla = (
        edad
        .value_counts()
        .sort_index()
        .rename_axis("Edad")
        .reset_index(name="Total")
    )

    tabla["Porcentaje"] = (
        tabla["Total"] / tabla["Total"].sum() * 100
    )

    tabla["Porcentaje"] = tabla["Porcentaje"].round(2)

    fila_total = pd.DataFrame({
        "Edad": ["TOTAL"],
        "Total": [tabla["Total"].sum()],
        "Porcentaje": [100.00]
    })

    tabla = pd.concat(
        [tabla, fila_total],
        ignore_index=True
    )

    return tabla


# ------------------------------------------------------------
# 10. GUARDAR RESULTADOS EN OBJETO
# ------------------------------------------------------------

resultados = {}

variables_categoricas = [
    "Sexo",
    "AfroMexicanos",
    "CondicionIndigena",
    "Escolaridad",
    "Ocupacion",
    "Ocupacion_Ampliado",
    "EstadoCivil"
]


# ============================================================
# 11. ESTADÍSTICAS GENERALES
# ============================================================

resultados["GENERAL"] = {}

for variable in variables_categoricas:

    resultados["GENERAL"][variable] = (
        tabla_categorica(datos, variable)
    )

resultados["GENERAL"]["Edad"] = {
    "estadisticas": estadisticas_edad(datos),
    "distribucion": tabla_edades(datos)
}


# ============================================================
# 12. ESTADÍSTICAS POR CAUSA
# ============================================================

for causa in causas:

    df_causa = datos[
        datos["Causa"] == causa
    ].copy()

    resultados[causa] = {
        "N": len(df_causa)
    }

    # Variables categóricas
    for variable in variables_categoricas:

        resultados[causa][variable] = (
            tabla_categorica(
                df_causa,
                variable
            )
        )

    # Edad
    resultados[causa]["Edad"] = {
        "estadisticas": estadisticas_edad(
            df_causa
        ),
        "distribucion": tabla_edades(
            df_causa
        )
    }


# ============================================================
# 13. GENERAR REPORTE FINAL
# ============================================================

print("\n")
print("=" * 80)
print("REPORTE ESTADÍSTICO DESCRIPTIVO")
print("=" * 80)


# ------------------------------------------------------------
# REPORTE GENERAL
# ------------------------------------------------------------

print("\n")
print("1. ESTADÍSTICAS GENERALES")
print("-" * 80)

print(f"\nTotal general de registros: {N_TOTAL:,}")
print(f"Número de causas: {numero_causas}")


for variable in variables_categoricas:

    print("\n")
    print(f"--- {variable} ---")

    display(
        resultados["GENERAL"][variable]
        .style
        .format({
            "Porcentaje": "{:.2f}%"
        })
    )


# ------------------------------------------------------------
# EDAD GENERAL
# ------------------------------------------------------------

print("\n")
print("--- EDAD: ESTADÍSTICOS DESCRIPTIVOS ---")

resumen_edad, conclusion = (
    resultados["GENERAL"]["Edad"]["estadisticas"]
)

display(
    resumen_edad.style.format({
        "Valor": "{:.4f}"
    })
)

print(
    f"Interpretación de normalidad: {conclusion}"
)

print("\n--- DISTRIBUCIÓN DE EDAD ---")

display(
    resultados["GENERAL"]["Edad"]["distribucion"]
    .style
    .format({
        "Porcentaje": "{:.2f}%"
    })
)


# ============================================================
# 14. REPORTE POR CAUSA
# ============================================================

print("\n\n")
print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS POR CAUSA")
print("=" * 80)


for i, causa in enumerate(causas, start=1):

    df_causa = datos[
        datos["Causa"] == causa
    ]

    print("\n")
    print("=" * 80)
    print(f"CAUSA {i}")
    print("=" * 80)

    print(f"\n{causa}")
    print(f"\nTotal de defunciones: {len(df_causa):,}")


    # --------------------------------------------------------
    # Variables categóricas
    # --------------------------------------------------------

    for variable in variables_categoricas:

        print("\n")
        print(f"--- {variable} ---")

        display(
            resultados[causa][variable]
            .style
            .format({
                "Porcentaje": "{:.2f}%"
            })
        )


    # --------------------------------------------------------
    # Edad
    # --------------------------------------------------------

    print("\n")
    print("--- EDAD: ESTADÍSTICOS DESCRIPTIVOS ---")

    resumen_edad, conclusion = (
        resultados[causa]["Edad"]["estadisticas"]
    )

    display(
        resumen_edad.style.format({
            "Valor": "{:.4f}"
        })
    )

    print(
        f"Interpretación de normalidad: {conclusion}"
    )


    print("\n--- DISTRIBUCIÓN DE EDAD ---")

    display(
        resultados[causa]["Edad"]["distribucion"]
        .style
        .format({
            "Porcentaje": "{:.2f}%"
        })
    )


# ============================================================
# 15. RESUMEN DE TAMAÑOS DE MUESTRA
# ============================================================

print("\n\n")
print("=" * 80)
print("RESUMEN DE LA MUESTRA")
print("=" * 80)

resumen_muestra = (
    datos["Causa"]
    .value_counts()
    .rename_axis("Causa")
    .reset_index(name="Total")
)

resumen_muestra["Porcentaje"] = (
    resumen_muestra["Total"]
    / resumen_muestra["Total"].sum()
    * 100
)

resumen_muestra["Porcentaje"] = (
    resumen_muestra["Porcentaje"].round(2)
)

fila_total = pd.DataFrame({
    "Causa": ["TOTAL GENERAL"],
    "Total": [resumen_muestra["Total"].sum()],
    "Porcentaje": [100.00]
})

resumen_muestra = pd.concat(
    [resumen_muestra, fila_total],
    ignore_index=True
)

display(
    resumen_muestra.style.format({
        "Porcentaje": "{:.2f}%"
    })
)

print("\n")
print("=" * 80)
print("FIN DEL REPORTE ESTADÍSTICO")
print("=" * 80)

Selecciona el archivo CSV...


Saving Defunciones.csv to Defunciones.csv

Archivo cargado: Defunciones.csv
Codificación utilizada: latin1


REPORTE ESTADÍSTICO DESCRIPTIVO


1. ESTADÍSTICAS GENERALES
--------------------------------------------------------------------------------

Total general de registros: 75,603
Número de causas: 3


--- Sexo ---


,Categoría,Total,Porcentaje
0,Hombre,44272,58.56%
1,Mujer,31331,41.44%
2,TOTAL,75603,100.00%




--- AfroMexicanos ---


,Categoría,Total,Porcentaje
0,No,66988,88.60%
1,No aplica,4552,6.02%
2,No especificado,3566,4.72%
3,Si,497,0.66%
4,TOTAL,75603,100.00%


...

Interpretación de normalidad: Se rechaza la normalidad

--- DISTRIBUCIÓN DE EDAD ---


,Edad,Total,Porcentaje
0,1,1,0.01%
1,2,1,0.01%
2,3,3,0.03%
3,4,1,0.01%
4,6,4,0.03%
5,7,2,0.02%
6,8,6,0.05%
7,10,6,0.05%
8,11,6,0.05%
9,12,1,0.01%





RESUMEN DE LA MUESTRA


,Causa,Total,Porcentaje
0,"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",35843,47.41%
1,"Neumonia, no especificada",28030,37.08%
2,"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",11730,15.52%
3,TOTAL GENERAL,75603,100.00%




FIN DEL REPORTE ESTADÍSTICO


In [ ]:
# ============================================================
# ANÁLISIS DE JI-CUADRADA DE INDEPENDENCIA
# CAUSA vs VARIABLES CATEGÓRICAS
#
# Incluye:
#   1. Chi-cuadrada global
#   2. p-valor
#   3. V de Cramér
#   4. Tablas de contingencia
#   5. Residuales estandarizados ajustados
#   6. Identificación de las categorías asociadas
#   7. Corrección por comparaciones múltiples
#
# NO GUARDA ARCHIVOS
# ============================================================


# ============================================================
# 1. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    norm
)

from statsmodels.stats.multitest import multipletests

from google.colab import files

from IPython.display import display

import io
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 2. CARGAR ARCHIVO
# ============================================================

print("Selecciona el archivo CSV...")

uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]

print(f"\nArchivo cargado: {nombre_archivo}")


# ============================================================
# 3. LEER ARCHIVO
# ============================================================

codificaciones = [
    "utf-8",
    "latin1",
    "cp1252"
]

datos = None

for enc in codificaciones:

    try:

        datos = pd.read_csv(
            io.BytesIO(
                uploaded[nombre_archivo]
            ),
            encoding=enc
        )

        print(
            f"Codificación utilizada: {enc}"
        )

        break

    except UnicodeDecodeError:

        continue


if datos is None:

    raise ValueError(
        "No fue posible leer el archivo."
    )


# ============================================================
# 4. LIMPIEZA BÁSICA
# ============================================================

datos.columns = (
    datos.columns
    .str.strip()
)


columnas_texto = (
    datos
    .select_dtypes(include="object")
    .columns
)


for col in columnas_texto:

    datos[col] = (
        datos[col]
        .astype(str)
        .str.strip()
    )


datos = datos.replace(
    [
        "nan",
        "NaN",
        "None",
        ""
    ],
    np.nan
)


# ============================================================
# 5. VARIABLES CATEGÓRICAS
# ============================================================

columnas_categoricas = [

    "Sexo",

    "AfroMexicanos",

    "CondicionIndigena",

    "Escolaridad",

    "Ocupacion",

    "Ocupacion_Ampliado",

    "EstadoCivil"

]


# ============================================================
# 6. VERIFICAR COLUMNAS
# ============================================================

columnas_necesarias = [
    "Causa"
] + columnas_categoricas


faltantes = [
    col
    for col in columnas_necesarias
    if col not in datos.columns
]


if faltantes:

    raise ValueError(
        "Faltan las siguientes columnas:\n"
        +
        "\n".join(
            f"- {x}"
            for x in faltantes
        )
    )


# ============================================================
# 7. CONFIGURACIÓN
# ============================================================

ALFA = 0.05


# ============================================================
# 8. FUNCIÓN V DE CRAMÉR
# ============================================================

def cramers_v(
    tabla,
    chi2
):

    n = tabla.to_numpy().sum()

    filas, columnas = (
        tabla.shape
    )

    minimo = min(
        filas - 1,
        columnas - 1
    )

    if (
        minimo <= 0
        or n <= 0
    ):

        return np.nan

    return np.sqrt(
        chi2 /
        (
            n * minimo
        )
    )


# ============================================================
# 9. INTERPRETACIÓN DE V DE CRAMÉR
# ============================================================

def interpretar_v(v):

    if pd.isna(v):

        return "No calculable"

    elif v < 0.10:

        return "Muy débil"

    elif v < 0.20:

        return "Débil"

    elif v < 0.30:

        return "Moderada"

    elif v < 0.50:

        return "Fuerte"

    else:

        return "Muy fuerte"


# ============================================================
# 10. FUNCIÓN PARA ANALIZAR LAS CELDAS
# ============================================================

def analizar_celdas(
    tabla,
    esperados
):

    observados = (
        tabla
        .to_numpy()
        .astype(float)
    )


    # --------------------------------------------------------
    # Totales
    # --------------------------------------------------------

    totales_fila = (
        observados.sum(
            axis=1,
            keepdims=True
        )
    )

    totales_columna = (
        observados.sum(
            axis=0,
            keepdims=True
        )
    )

    n = observados.sum()


    # --------------------------------------------------------
    # Residuales estandarizados ajustados
    # --------------------------------------------------------

    proporciones_fila = (
        totales_fila / n
    )

    proporciones_columna = (
        totales_columna / n
    )


    # Residual ajustado:

    residuales = (
        (
            observados
            - esperados
        )
        /
        np.sqrt(
            esperados
            *
            (
                1
                - proporciones_fila
            )
            *
            (
                1
                - proporciones_columna
            )
        )
    )


    # --------------------------------------------------------
    # p-valor de cada celda
    # --------------------------------------------------------

    p_celdas = (
        2
        *
        norm.sf(
            np.abs(residuales)
        )
    )


    # --------------------------------------------------------
    # Crear dataframe
    # --------------------------------------------------------

    filas = []

    nombres_filas = (
        tabla.index.tolist()
    )

    nombres_columnas = (
        tabla.columns.tolist()
    )


    for i, causa in enumerate(
        nombres_filas
    ):

        for j, categoria in enumerate(
            nombres_columnas
        ):

            observado = (
                observados[i, j]
            )

            esperado = (
                esperados[i, j]
            )

            residual = (
                residuales[i, j]
            )

            p = (
                p_celdas[i, j]
            )


            # Porcentaje de la categoría
            # dentro de la causa

            total_causa = (
                observados[i, :].sum()
            )

            porcentaje_fila = (
                observado
                /
                total_causa
                *
                100
            )


            # Dirección de la asociación

            if residual > 0:

                direccion = (
                    "Mayor de lo esperado"
                )

            elif residual < 0:

                direccion = (
                    "Menor de lo esperado"
                )

            else:

                direccion = (
                    "Igual a lo esperado"
                )


            filas.append({

                "Causa":
                    causa,

                "Categoría":
                    categoria,

                "Observado":
                    observado,

                "Esperado":
                    esperado,

                "Residual":
                    residual,

                "p_celda":
                    p,

                "Porcentaje_dentro_Causa":
                    porcentaje_fila,

                "Dirección":
                    direccion

            })


    resultado = pd.DataFrame(
        filas
    )


    # --------------------------------------------------------
    # Corrección de Holm
    # --------------------------------------------------------

    if len(resultado) > 0:

        (
            rechazo,
            p_ajustado,
            _,
            _
        ) = multipletests(
            resultado["p_celda"],
            alpha=ALFA,
            method="holm"
        )

        resultado[
            "p_ajustado"
        ] = p_ajustado

        resultado[
            "Significativa"
        ] = rechazo

    else:

        resultado[
            "p_ajustado"
        ] = []

        resultado[
            "Significativa"
        ] = []


    return resultado


# ============================================================
# 11. FUNCIÓN PRINCIPAL DE CHI-CUADRADA
# ============================================================

def prueba_chi_cuadrada(
    df,
    variable
):

    # --------------------------------------------------------
    # Seleccionar datos válidos
    # --------------------------------------------------------

    temp = df[
        [
            "Causa",
            variable
        ]
    ].dropna()


    # --------------------------------------------------------
    # Tabla de contingencia
    # --------------------------------------------------------

    tabla = pd.crosstab(
        temp["Causa"],
        temp[variable]
    )


    # --------------------------------------------------------
    # Verificar dimensiones
    # --------------------------------------------------------

    if (
        tabla.shape[0] < 2
        or tabla.shape[1] < 2
    ):

        return None


    # --------------------------------------------------------
    # Chi-cuadrada
    # --------------------------------------------------------

    (
        chi2,
        p,
        gl,
        esperados
    ) = chi2_contingency(
        tabla,
        correction=False
    )


    # --------------------------------------------------------
    # V de Cramér
    # --------------------------------------------------------

    v = cramers_v(
        tabla,
        chi2
    )


    # --------------------------------------------------------
    # Frecuencias esperadas
    # --------------------------------------------------------

    esperados = np.array(
        esperados
    )


    porcentaje_menor_5 = (
        (
            esperados < 5
        ).sum()
        /
        esperados.size
        *
        100
    )


    minimo_esperado = (
        esperados.min()
    )


    # --------------------------------------------------------
    # Analizar celdas
    # --------------------------------------------------------

    celdas = analizar_celdas(
        tabla,
        esperados
    )


    # --------------------------------------------------------
    # Resultado global
    # --------------------------------------------------------

    resultado = {

        "Variable":
            variable,

        "N":
            len(temp),

        "Chi2":
            chi2,

        "gl":
            gl,

        "p":
            p,

        "V_Cramer":
            v,

        "Interpretacion_V":
            interpretar_v(v),

        "Celdas_esperadas_<5_%":
            porcentaje_menor_5,

        "Min_esperado":
            minimo_esperado,

        "Significativa":
            p < ALFA

    }


    return (
        resultado,
        tabla,
        celdas
    )


# ============================================================
# 12. EJECUTAR TODAS LAS PRUEBAS
# ============================================================

resultados_chi = []

tablas_contingencia = {}

resultados_celdas = {}


for variable in columnas_categoricas:

    resultado = prueba_chi_cuadrada(
        datos,
        variable
    )


    if resultado is None:

        continue


    (
        resultado_dict,
        tabla,
        celdas
    ) = resultado


    resultados_chi.append(
        resultado_dict
    )


    tablas_contingencia[
        variable
    ] = tabla


    resultados_celdas[
        variable
    ] = celdas


# ============================================================
# 13. DATAFRAME GENERAL
# ============================================================

resultados_chi = pd.DataFrame(
    resultados_chi
)


# ============================================================
# 14. ORDENAR POR P-VALOR
# ============================================================

resultados_chi = (
    resultados_chi
    .sort_values(
        by="p"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 15. SELECCIONAR SOLAMENTE LAS SIGNIFICATIVAS
# ============================================================

significativas = resultados_chi[
    resultados_chi[
        "Significativa"
    ]
].copy()


# ============================================================
# 16. REPORTE GENERAL
# ============================================================

print("\n")

print("=" * 100)

print(
    "ANÁLISIS DE INDEPENDENCIA"
)

print(
    "CAUSA vs VARIABLES CATEGÓRICAS"
)

print("=" * 100)


print(
    f"\nNivel de significancia: "
    f"α = {ALFA}"
)


print(
    f"\nNúmero de pruebas realizadas: "
    f"{len(resultados_chi)}"
)


print(
    f"Número de asociaciones significativas: "
    f"{len(significativas)}"
)


# ============================================================
# 17. TABLA GENERAL DE ASOCIACIONES SIGNIFICATIVAS
# ============================================================

print("\n")

print("=" * 100)

print(
    "1. ASOCIACIONES SIGNIFICATIVAS"
)

print("=" * 100)


if len(significativas) == 0:

    print(
        "\nNo se encontraron asociaciones "
        "estadísticamente significativas."
    )

else:

    tabla_resumen = significativas[
        [
            "Variable",
            "N",
            "Chi2",
            "gl",
            "p",
            "V_Cramer",
            "Interpretacion_V",
            "Celdas_esperadas_<5_%",
            "Min_esperado"
        ]
    ].copy()


    tabla_resumen = (
        tabla_resumen
        .rename(
            columns={

                "Chi2":
                    "χ²",

                "V_Cramer":
                    "V de Cramér",

                "Interpretacion_V":
                    "Magnitud",

                "Celdas_esperadas_<5_%":
                    "Celdas esperadas <5 (%)",

                "Min_esperado":
                    "Mínimo esperado"

            }
        )
    )


    display(
        tabla_resumen.style.format({

            "N":
                "{:,.0f}",

            "χ²":
                "{:,.3f}",

            "p":
                "{:.6g}",

            "V de Cramér":
                "{:.3f}",

            "Celdas esperadas <5 (%)":
                "{:.2f}%",

            "Mínimo esperado":
                "{:.3f}"

        })
    )


# ============================================================
# 18. ANÁLISIS DETALLADO DE CADA VARIABLE SIGNIFICATIVA
# ============================================================

print("\n")

print("=" * 100)

print(
    "2. ¿QUÉ CATEGORÍAS ESTÁN RELACIONADAS CON CADA CAUSA?"
)

print("=" * 100)


for _, fila in significativas.iterrows():

    variable = fila[
        "Variable"
    ]


    print("\n")

    print("#" * 100)

    print(
        f"CAUSA × {variable}"
    )

    print("#" * 100)


    print(
        f"\nχ² = "
        f"{fila['Chi2']:.3f}"
    )

    print(
        f"gl = "
        f"{int(fila['gl'])}"
    )

    print(
        f"p = "
        f"{fila['p']:.6g}"
    )

    print(
        f"V de Cramér = "
        f"{fila['V_Cramer']:.3f}"
    )

    print(
        f"Magnitud = "
        f"{fila['Interpretacion_V']}"
    )


    # --------------------------------------------------------
    # Tabla de contingencia
    # --------------------------------------------------------

    print(
        "\nTabla de frecuencias observadas:"
    )


    tabla = (
        tablas_contingencia[
            variable
        ]
        .copy()
    )


    tabla["TOTAL"] = (
        tabla.sum(
            axis=1
        )
    )


    fila_total = (
        tabla.sum(
            axis=0
        )
        .to_frame()
        .T
    )


    fila_total.index = [
        "TOTAL"
    ]


    tabla = pd.concat(
        [
            tabla,
            fila_total
        ]
    )


    display(
        tabla
    )


    # --------------------------------------------------------
    # Todas las celdas con sus residuales
    # --------------------------------------------------------

    celdas = (
        resultados_celdas[
            variable
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Solamente categorías significativas
    # después de corrección de Holm
    # --------------------------------------------------------

    celdas_significativas = celdas[
        celdas[
            "Significativa"
        ]
    ].copy()


    # --------------------------------------------------------
    # Ordenar por magnitud absoluta del residual
    # --------------------------------------------------------

    celdas_significativas[
        "Abs_Residual"
    ] = np.abs(
        celdas_significativas[
            "Residual"
        ]
    )


    celdas_significativas = (
        celdas_significativas
        .sort_values(
            by="Abs_Residual",
            ascending=False
        )
    )


    print(
        "\nCategorías específicamente asociadas:"
    )


    if len(
        celdas_significativas
    ) == 0:

        print(
            "\nNinguna celda individual "
            "resultó significativa después "
            "de la corrección de Holm."
        )

    else:

        tabla_categorias = (
            celdas_significativas[
                [
                    "Causa",
                    "Categoría",
                    "Observado",
                    "Esperado",
                    "Residual",
                    "p_celda",
                    "p_ajustado",
                    "Porcentaje_dentro_Causa",
                    "Dirección"
                ]
            ]
            .copy()
        )


        tabla_categorias = (
            tabla_categorias
            .rename(
                columns={

                    "Observado":
                        "Observado",

                    "Esperado":
                        "Esperado",

                    "Residual":
                        "Residual ajustado",

                    "p_celda":
                        "p celda",

                    "p_ajustado":
                        "p ajustado",

                    "Porcentaje_dentro_Causa":
                        "% dentro de la causa",

                    "Dirección":
                        "Relación"

                }
            )
        )


        display(
            tabla_categorias.style.format({

                "Observado":
                    "{:,.0f}",

                "Esperado":
                    "{:,.2f}",

                "Residual ajustado":
                    "{:.3f}",

                "p celda":
                    "{:.6g}",

                "p ajustado":
                    "{:.6g}",

                "% dentro de la causa":
                    "{:.2f}%"

            })
        )


        # ----------------------------------------------------
        # Explicación textual de las categorías
        # ----------------------------------------------------

        print(
            "\nInterpretación de las categorías:"
        )


        for _, celda in (
            celdas_significativas
            .iterrows()
        ):

            causa = celda[
                "Causa"
            ]

            categoria = celda[
                "Categoría"
            ]

            observado = celda[
                "Observado"
            ]

            esperado = celda[
                "Esperado"
            ]

            residual = celda[
                "Residual"
            ]

            porcentaje = celda[
                "Porcentaje_dentro_Causa"
            ]

            if residual > 0:

                print(
                    f"\n• {causa}"
                    f" — {variable}: "
                    f"'{categoria}'"
                    f" → aparece MÁS veces "
                    f"de lo esperado."
                    f" Observado = "
                    f"{observado:,.0f};"
                    f" esperado = "
                    f"{esperado:,.2f};"
                    f" {porcentaje:.2f}% "
                    f"dentro de esta causa."
                )

            elif residual < 0:

                print(
                    f"\n• {causa}"
                    f" — {variable}: "
                    f"'{categoria}'"
                    f" → aparece MENOS veces "
                    f"de lo esperado."
                    f" Observado = "
                    f"{observado:,.0f};"
                    f" esperado = "
                    f"{esperado:,.2f};"
                    f" {porcentaje:.2f}% "
                    f"dentro de esta causa."
                )


# ============================================================
# 19. RESUMEN FINAL POR CAUSA
# ============================================================

print("\n")

print("=" * 100)

print(
    "3. RESUMEN: CATEGORÍAS ASOCIADAS A CADA CAUSA"
)

print("=" * 100)


if len(significativas) == 0:

    print(
        "\nNo existen asociaciones significativas "
        "para resumir."
    )

else:

    for causa in datos[
        "Causa"
    ].dropna().unique():

        print("\n")

        print("-" * 100)

        print(
            f"CAUSA: {causa}"
        )

        print("-" * 100)


        encontro = False


        for variable in columnas_categoricas:

            if variable not in resultados_celdas:

                continue


            celdas = resultados_celdas[
                variable
            ]


            celdas_causa = celdas[
                (
                    celdas["Causa"]
                    == causa
                )
                &
                (
                    celdas["Significativa"]
                    == True
                )
            ]


            if len(
                celdas_causa
            ) == 0:

                continue


            # Sólo incluimos la variable si
            # su Chi-cuadrada global fue significativa

            resultado_variable = (
                significativas[
                    significativas[
                        "Variable"
                    ]
                    == variable
                ]
            )


            if len(
                resultado_variable
            ) == 0:

                continue


            encontro = True


            print(
                f"\n{variable}:"
            )


            for _, celda in (
                celdas_causa
                .sort_values(
                    "Residual",
                    ascending=False
                )
                .iterrows()
            ):

                categoria = celda[
                    "Categoría"
                ]

                residual = celda[
                    "Residual"
                ]

                observado = celda[
                    "Observado"
                ]

                esperado = celda[
                    "Esperado"
                ]


                if residual > 0:

                    print(
                        f"   ↑ {categoria}"
                        f" | observado: "
                        f"{observado:,.0f}"
                        f" | esperado: "
                        f"{esperado:,.2f}"
                        f" | residual: "
                        f"{residual:.2f}"
                    )

                else:

                    print(
                        f"   ↓ {categoria}"
                        f" | observado: "
                        f"{observado:,.0f}"
                        f" | esperado: "
                        f"{esperado:,.2f}"
                        f" | residual: "
                        f"{residual:.2f}"
                    )


        if not encontro:

            print(
                "\nNo se identificaron "
                "categorías individuales "
                "significativamente asociadas "
                "con esta causa."
            )


# ============================================================
# 20. RESUMEN GENERAL
# ============================================================

print("\n")

print("=" * 100)

print(
    "ANÁLISIS TERMINADO"
)

print("=" * 100)

print(
    "\nSe analizaron todas las variables categóricas "
    "contra Causa."
)

print(
    "El reporte principal conserva únicamente "
    "las asociaciones globales con p < 0.05."
)

print(
    "\nPara las asociaciones significativas, "
    "se identificaron las categorías específicas "
    "mediante residuales estandarizados ajustados "
    "y corrección de Holm por comparaciones múltiples."
)

print(
    "\nNo se generaron ni guardaron archivos."
)

Selecciona el archivo CSV...


Saving Defunciones.csv to Defunciones (2).csv

Archivo cargado: Defunciones (2).csv
Codificación utilizada: latin1


ANÁLISIS DE INDEPENDENCIA
CAUSA vs VARIABLES CATEGÓRICAS

Nivel de significancia: α = 0.05

Número de pruebas realizadas: 7
Número de asociaciones significativas: 7


1. ASOCIACIONES SIGNIFICATIVAS


,Variable,N,χ²,gl,p,V de Cramér,Magnitud,Celdas esperadas <5 (%),Mínimo esperado
0,Sexo,"75,603","6,512.978",2,0,0.294,Moderada,0.00%,4861.085
1,Escolaridad,"75,603","11,073.971",22,0,0.271,Moderada,0.00%,18.463
2,Ocupacion_Ampliado,"75,603","17,578.302",110,0,0.341,Fuerte,17.26%,0.310
3,Ocupacion,"75,603","9,075.479",22,0,0.245,Moderada,0.00%,7.602
4,EstadoCivil,"75,603","11,378.856",14,0,0.274,Moderada,0.00%,157.170
5,CondicionIndigena,"75,603",969.957,6,2.80941e-206,0.080,Muy débil,0.00%,608.043
6,AfroMexicanos,"75,603",387.380,6,1.44244e-80,0.051,Muy débil,0.00%,77.111




2. ¿QUÉ CATEGORÍAS ESTÁN RELACIONADAS CON CADA CAUSA?


####################################################################################################
CAUSA × Sexo
####################################################################################################

χ² = 6512.978
gl = 2
p = 0
V de Cramér = 0.294
Magnitud = Moderada

Tabla de frecuencias observadas:


Sexo,Hombre,Mujer,TOTAL
"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",10791,939,11730
"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",18121,17722,35843
"Neumonia, no especificada",15360,12670,28030
TOTAL,44272,31331,75603


...


Interpretación de las categorías:

• Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras — CondicionIndigena: 'No especificado' → aparece MÁS veces de lo esperado. Observado = 1,172; esperado = 608.04; 9.99% dentro de esta causa.

• Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras — CondicionIndigena: 'Si' → aparece MENOS veces de lo esperado. Observado = 289; esperado = 645.75; 2.46% dentro de esta causa.

• Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas — CondicionIndigena: 'No especificado' → aparece MENOS veces de lo esperado. Observado = 1,464; esperado = 1,857.98; 4.08% dentro de esta causa.

• Neumonia, no especificada — CondicionIndigena: 'No aplica' → aparece MENOS veces de lo esperado. Observado = 1,373; esperado = 1,687.67; 4.90% dentro de esta causa.

• Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas — CondicionIndigena: 'No aplica' → aparece MÁS veces de lo es

AfroMexicanos,No,No aplica,No especificado,Si,TOTAL
"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",10046,725,902,57,11730
"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",31635,2454,1508,246,35843
"Neumonia, no especificada",25307,1373,1156,194,28030
TOTAL,66988,4552,3566,497,75603



Categorías específicamente asociadas:


,Causa,Categoría,Observado,Esperado,Residual ajustado,p celda,p ajustado,% dentro de la causa,Relación
2,"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",No especificado,902,553.27,16.524,2.46269e-61,2.95523e-60,7.69%,Mayor de lo esperado
8,"Neumonia, no especificada",No,"25,307","24,835.97",11.162,6.25758e-29,6.88334e-28,90.29%,Mayor de lo esperado
0,"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",No,"10,046","10,393.36",-10.981,4.69911e-28,4.69911e-27,85.64%,Menor de lo esperado
9,"Neumonia, no especificada",No aplica,"1,373","1,687.67",-9.960,2.27012e-23,2.04311e-22,4.90%,Menor de lo esperado
5,"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",No aplica,"2,454","2,158.08",9.061,1.29387e-19,1.03509e-18,6.85%,Mayor de lo esperado
6,"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",No especificado,"1,508","1,690.62",-6.274,3.51085e-10,2.45759e-09,4.21%,Menor de lo esperado
10,"Neumonia, no especificada",No especificado,"1,156","1,322.10",-5.900,3.64261e-09,2.18557e-08,4.12%,Menor de lo esperado
4,"Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas",No,"31,635","31,758.67",-2.835,0.00458486,0.0229243,88.26%,Menor de lo esperado
3,"Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras",Si,57,77.11,-2.500,0.0124243,0.0496972,0.49%,Menor de lo esperado



Interpretación de las categorías:

• Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras — AfroMexicanos: 'No especificado' → aparece MÁS veces de lo esperado. Observado = 902; esperado = 553.27; 7.69% dentro de esta causa.

• Neumonia, no especificada — AfroMexicanos: 'No' → aparece MÁS veces de lo esperado. Observado = 25,307; esperado = 24,835.97; 90.29% dentro de esta causa.

• Agresion Con Disparo De Otras Armas De Fuego, Y Las No Especificadas, Calles Y Carreteras — AfroMexicanos: 'No' → aparece MENOS veces de lo esperado. Observado = 10,046; esperado = 10,393.36; 85.64% dentro de esta causa.

• Neumonia, no especificada — AfroMexicanos: 'No aplica' → aparece MENOS veces de lo esperado. Observado = 1,373; esperado = 1,687.67; 4.90% dentro de esta causa.

• Diabetes Mellitus Tipo 2, Con Otras Complicaciones Especificadas — AfroMexicanos: 'No aplica' → aparece MÁS veces de lo esperado. Observado = 2,454; esperado = 2,158.08; 6.85% dentro de es